# Ramp Resource LP — EFHK Validation Notebook

Validates all nine user stories against the EFHK reference dataset (`data/finavia_flights_efhk_20260327.csv`).

| Stage | Function | User story |
|-------|----------|------------|
| 1 | `compute_demand()` | US1–US3, US6–US8 |
| 2 | `schedule_shifts()` | US4–US5 |
| Analysis | `identify_bottlenecks()` | US9 |
| Analysis | `comparison_report()` | FR-008 |

In [1]:
import sys
sys.path.insert(0, '../..')  # ensure src/ is importable from notebooks/planning/

from src.utils import load_efhk
from src.lp import (
    compute_demand, schedule_shifts,
    identify_bottlenecks, comparison_report,
    AircraftType,
)

## 1  Load EFHK Dataset

In [2]:
DATA = '../../data/finavia_flights_efhk_20260327.csv'

scheduled, movements = load_efhk(DATA)
print(f'Scheduled slots : {len(scheduled)}')
print(f'Movement records: {len(movements) if movements else 0}')

Scheduled slots : 18
Movement records: 381


## 2  Stage 1 — Demand Curve

In [3]:
# US8: use tau_movements for tolerance-window reclassification (FR-011)
demand = compute_demand(scheduled, tau_movements=movements)

print(f'Feasible : {demand.feasible}')
print(f'Infeasible slots: {demand.infeasible_slots}')
print()
print(f'{"Hour":<6} {"Demand":>7} {"Arr":>5} {"Dep":>5}')
print('-' * 30)
for h, d, a, dp in zip(
    demand.operating_hours,
    demand.demand_curve,
    demand.arrival_demand_curve,
    demand.departure_demand_curve,
):
    print(f'{h:02d}:00  {d:>7}  {a:>5}  {dp:>5}')

Feasible : True
Infeasible slots: []

Hour    Demand   Arr   Dep
------------------------------
05:00       19     15      4
06:00       30     12     18
07:00       60      6     54
08:00       52     30     22
09:00       39     24     15
10:00       41     21     20
11:00       45     18     27
12:00       32     16     16
13:00       35     17     18
14:00       28     22      6
15:00       63     49     14
16:00       66     14     52
17:00       60     16     44
18:00       25      9     16
19:00       29     14     15
20:00       38     20     18
21:00       42     26     16
22:00       48     34     14


## 3  Stage 2 — Minimum Shift Schedule

In [4]:
schedule = schedule_shifts(demand)

print(f'Daily headcount  : {schedule.daily_headcount}')
print(f'Coverage satisfied: {schedule.coverage_satisfied}')
print(f'Coverage shortfalls: {schedule.coverage_shortfalls}')
print()
print(f'{"Start hour":<12} {"Shifts (rounded)":>16}')
print('-' * 30)
for h, n in sorted(schedule.shift_starts_rounded.items()):
    if n > 0:
        print(f'{h:02d}:00        {n:>16}')

Daily headcount  : 121
Coverage satisfied: True
Coverage shortfalls: []

Start hour   Shifts (rounded)
------------------------------
05:00                      54
07:00                       7
15:00                      60


## 4  Bottleneck Hours (US9)

In [5]:
bottlenecks = identify_bottlenecks(demand, schedule)

if bottlenecks.bottleneck_hours:
    print('Bottleneck hours (active workers == demand):')
    for h in bottlenecks.bottleneck_hours:
        print(f'  {h:02d}:00  demand={bottlenecks.demand_at_bottleneck[h]}')
else:
    print('No bottleneck hours — all slots have surplus workers.')

Bottleneck hours (active workers == demand):
  06:00  demand=26
  07:00  demand=60


## 5  Scheduled vs Tau Comparison (FR-008)

In [6]:
tau_slots, _ = load_efhk(DATA, use_tau_times=True, extract_tau=False)
report = comparison_report(scheduled, tau_slots)

print(f'Arrival gap   : {report.arrival_gap_pct_total:+.1f}%')
print(f'Departure gap : {report.departure_gap_pct_total:+.1f}%')
print()
print(f'{"Hour":<6} {"S-Arr":>6} {"T-Arr":>6} {"Gap":>6} | {"S-Dep":>6} {"T-Dep":>6} {"Gap":>6}')
print('-' * 50)
for i, h in enumerate(report.hours):
    print(
        f'{h:02d}:00  '
        f'{report.scheduled_arrival_demand[i]:>6}  '
        f'{report.tau_arrival_demand[i]:>6}  '
        f'{report.arrival_gap_absolute[i]:>+6}  | '
        f'{report.scheduled_departure_demand[i]:>6}  '
        f'{report.tau_departure_demand[i]:>6}  '
        f'{report.departure_gap_absolute[i]:>+6}'
    )

Arrival gap   : -1.0%
Departure gap : +0.0%

Hour    S-Arr  T-Arr    Gap |  S-Dep  T-Dep    Gap
--------------------------------------------------
05:00      25      15     -10  |      4       4      +0
06:00       8       8      +0  |     18      21      +3
07:00      12       7      -5  |     54      50      -4
08:00      32      25      -7  |     28      22      -6
09:00      16      23      +7  |     21      16      -5
10:00      10      17      +7  |     14      18      +4
11:00      12      12      +0  |     22      27      +5
12:00       7      12      +5  |     16      20      +4
13:00      12      11      -1  |     17      17      +0
14:00      19      17      -2  |      9       4      -5
15:00      39      34      -5  |     10      15      +5
16:00       9       9      +0  |     55      50      -5
17:00       4      12      +8  |     42      49      +7
18:00      13       6      -7  |     14      12      -2
19:00       9      10      +1  |     15      14      -1
20:00      16